# Elderly Care Agentic Vision System — Interactive Demo
This notebook demonstrates the end-to-end AI + Vision pipeline for elderly indoor monitoring in offline mock mode:
1. **Video Ingestion & Sampling**
2. **YOLOv8-Pose & Bed Overlap Estimation**
3. **Vision-Language Model (GPT-4o Vision / Offline Heuristics)**
4. **Temporal State Tracking & Debouncing**
5. **Agentic Disambiguation & Safety Alerts**
6. **Evaluation & Visualization**

In [ ]:
import json
import tempfile
from pathlib import Path
import matplotlib.pyplot as plt

api_key = "mock"
print("Running in offline mock mode; no OpenAI API calls will be made.")

## 1. Run the Pipeline
We can generate a synthetic demo video and run the entire pipeline in one call:

In [ ]:
from demo import create_synthetic_video
from pipeline import PipelineConfig, ElderlyCarePipeline

notebook_temp = tempfile.TemporaryDirectory(prefix="elderly-care-demo-")
notebook_dir = Path(notebook_temp.name)
video_path = str(notebook_dir / "demo_video.mp4")
output_dir = str(notebook_dir / "output")
create_synthetic_video(video_path, duration_sec=60, fps=10)

config = PipelineConfig(
    openai_api_key=api_key,
    vlm_model="gpt-4o",
    sample_interval_sec=2.0,
    output_dir=output_dir,
    enable_agentic_resolution=True,
    enable_pose_estimation=True,
    verbose=True,
)

pipeline = ElderlyCarePipeline(config)
report = pipeline.analyze(video_path)
print(f"\nAnalysis complete! Final alert: {report.final_alert.value}")

## 2. Visualize Activity Timeline and Durations

In [ ]:
with open(Path(output_dir) / "activity_summary.json") as f:
    summary = json.load(f)

with open(Path(output_dir) / "bed_events.json") as f:
    bed_events = json.load(f)

print("Activity Breakdown:")
for state, dur in summary["activity_summary"].items():
    print(f"  {state:<22} : {dur}")

print(f"\nBed Exits: {bed_events['bed_exit_count']} | Bed Returns: {bed_events['bed_return_count']}")
for ev in bed_events.get("events", []):
    print(f"  • [{ev['start_time']} -> {ev['confirmed_time']}] {ev['event'].upper()}: {ev['previous_state']} -> {ev['current_state']} (conf={ev['confidence']})")

## 3. Ground Truth Evaluation

In [ ]:
from evaluation import EvaluationMetrics

# Ground truth for the deterministic 60-second synthetic demo above.
gt = {
    "observation_duration_sec": 60,
    "activity_duration_sec": {
        "lying_in_bed": 27,
        "sitting_on_bed": 8,
        "sitting_outside_bed": 10,
        "standing": 2,
        "walking": 13,
        "out_of_bed": 0,
        "unknown": 0,
    },
    "bed_exit_count": 1,
    "bed_return_count": 1,
    "bed_exit_times_sec": [22],
    "bed_return_times_sec": [45],
    "event_tolerance_sec": 3,
    "timeline": [
        {"start_sec": 0, "end_sec": 15, "state": "lying_in_bed"},
        {"start_sec": 15, "end_sec": 20, "state": "sitting_on_bed"},
        {"start_sec": 20, "end_sec": 22, "state": "standing"},
        {"start_sec": 22, "end_sec": 30, "state": "walking"},
        {"start_sec": 30, "end_sec": 40, "state": "sitting_outside_bed"},
        {"start_sec": 40, "end_sec": 45, "state": "walking"},
        {"start_sec": 45, "end_sec": 48, "state": "sitting_on_bed"},
        {"start_sec": 48, "end_sec": 60, "state": "lying_in_bed"},
    ],
}
evaluator = EvaluationMetrics(report, gt)
metrics = evaluator.compute_all()
print(evaluator.format_report())